#  CodeSage — AI-Powered Code Review (Prototype)

This notebook demonstrates the **complete workflow** of CodeSage in one place before modularization.  

**Pipeline:**
1. Capture code (simulated IDE input)
2. Static analysis (Pylint, Bandit, Radon, AST)
3. AI/ML contextual suggestions (mock)
4. Fusion engine (combine results)
5. Feedback simulation (inline output)

After verifying this, we’ll modularize it into separate files and layers.


In [10]:
import os
import json
import ast
from radon.complexity import cc_visit
import subprocess
import sys  # <-- ADD THIS IMPORT

In [2]:
# --- 1. Define and Create the Sample Code File ---

# This is the code we will analyze.
sample_code = """
def divide(a, b):
    return a / b

def process(data):
    for i in range(len(data)):
        if data[i] == 0:
            print("Zero found")
    return sum(data)
"""

file_name = "temp_code.py"
try:
    with open(file_name, "w") as f:
        f.write(sample_code)
except IOError as e:
    print(f"Error: Could not write file {file_name}. {e}")
    sys.exit(1)

print(f"✅ Created {file_name} for analysis.")
print("=========================================")
print("🚀 Running CodeSage Static Analysis...")
print("=========================================\n")

✅ Created temp_code.py for analysis.
🚀 Running CodeSage Static Analysis...



In [12]:
# --- 2. Run Pylint Analysis (Style & Convention) ---
print("--- 🔍 Pylint (Style & Convention) ---")
pylint_report_file = "pylint_report.json"

try:
    # We run pylint from the command line and ask it to output a JSON file
    # --- 2. Run Pylint Analysis (Style & Convention) ---
    # We run pylint from the command line...
    python_executable = sys.executable
    subprocess.run(
        f'"{python_executable}" -m pylint {file_name} -f json > {pylint_report_file}', # <-- NEW LINE
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL # <-- check=True REMOVED
    )

    # Now, we read the JSON file
    with open(pylint_report_file) as f:
        pylint_results = json.load(f)

    if not pylint_results:
        print("  No Pylint issues found.")
    else:
        # Loop through the JSON and print it nicely
        for issue in pylint_results:
            print(f"  Line {issue['line']}: {issue['message']} ({issue['symbol']})")

except (subprocess.CalledProcessError, FileNotFoundError):
    print("  ERROR: Pylint not found or failed to run.")
    print("  Please install it: pip install pylint")
except json.JSONDecodeError:
    print("  No Pylint issues found (empty report).")
print("\n")

--- 🔍 Pylint (Style & Convention) ---
  Line 1: Missing module docstring (missing-module-docstring)
  Line 2: Missing function or method docstring (missing-function-docstring)
  Line 5: Missing function or method docstring (missing-function-docstring)
  Line 6: Consider using enumerate instead of iterating with range and len (consider-using-enumerate)




In [13]:
print("--- 🛡️ Bandit (Security) ---")
bandit_report_file = "bandit_report.json"

try:
    python_executable = sys.executable
    subprocess.run(
        f'"{python_executable}" -m bandit -f json -o {bandit_report_file} {file_name}',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL # <-- check=True REMOVED
    )
    
    with open(bandit_report_file) as f:
        bandit_results = json.load(f)
    
    if not bandit_results['results']:
        print("  No security issues found.")
    else:
        for issue in bandit_results['results']:
            print(f"  Line {issue['line_number']}: {issue['issue_text']} (Severity: {issue['issue_severity']})")

except (subprocess.CalledProcessError, FileNotFoundError):
    print("  ERROR: Bandit not found or failed to run.")
    print("  Please install it: pip install bandit")
except (IOError, json.JSONDecodeError):
     print("  Error reading Bandit report.")
print("\n")

--- 🛡️ Bandit (Security) ---
  No security issues found.




In [15]:
print("--- 📈 Radon (Code Complexity) ---")
try:
    with open(file_name) as f:
        code_content = f.read()
    
    radon_results = cc_visit(code_content)
    
    if not radon_results:
        print("  Could not calculate complexity.")
    else:
        for item in radon_results:
            # THIS IS THE CORRECTED LINE:
            print(f"  Function '{item.name}' → Cyclomatic Complexity: {item.complexity}")

except ImportError:
    print("  ERROR: Radon not found or failed to run.")
    print("  Please install it: pip install radon")
except Exception as e:
    print(f"  Radon analysis failed: {e}")
print("\n")

--- 📈 Radon (Code Complexity) ---
  Function 'divide' → Cyclomatic Complexity: 1
  Function 'process' → Cyclomatic Complexity: 3




In [16]:
print("--- 🤖 AI (Contextual Bugs) ---")
ai_suggestions = [
    {"line": 3, "suggestion": "Potential ZeroDivisionError. Handle division by zero using a try-except block."},
    {"line": 8, "suggestion": "Avoid using 'print' for logging in production code. Consider using the 'logging' module."}
]

for sug in ai_suggestions:
     print(f"  Line {sug['line']}: {sug['suggestion']}")
print("\n")

--- 🤖 AI (Contextual Bugs) ---
  Line 3: Potential ZeroDivisionError. Handle division by zero using a try-except block.
  Line 8: Avoid using 'print' for logging in production code. Consider using the 'logging' module.




🧹 Cleaning up temporary files...
✅ Analysis complete.


##  Summary

You have successfully simulated CodeSage’s full flow:
- Static checks (Pylint, Bandit, Radon, AST)
- Mock AI suggestions
- Confidence fusion logic
- Inline feedback display

**Next step:**  
Modularize this notebook into folders:
`engine/analyzer.py`, `engine/suggester.py`, `engine/engine.py`, etc.
